In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import gc
import cv2
import numpy as np
import os # para criar o diretório de saída
from dataloader import AntiUAVRGBTDataset, collate_fn_multimodal
from model import VisionTransformerMultimodal
import config

# ===============================
# Função de perda para GT caixas de detecção baseada em IoU.
# alpha: peso  do GIoU loss
# beta → peso do L1 loss sobre centro e tamanho.
# Exemplo: se alpha=1.0 e beta=0.5, a rede prioriza mais GIoU do que L1 para treinar caixas.
# ===============================
def combined_box_loss(pred_boxes, target_boxes, alpha=0.5, beta=1.5):
    # converter [cx,cy,w,h] -> [x1,y1,x2,y2]
    def cxcywh_to_xyxy(box):
        cx, cy, w, h = torch.chunk(box, 4, dim=-1)
        x1 = cx - 0.5 * w
        y1 = cy - 0.5 * h
        x2 = cx + 0.5 * w
        y2 = cy + 0.5 * h
        return torch.cat([x1, y1, x2, y2], dim=-1)

    pred_xyxy = cxcywh_to_xyxy(pred_boxes)
    gt_xyxy = cxcywh_to_xyxy(target_boxes)

    # interseção e união
    x1 = torch.max(pred_xyxy[..., 0], gt_xyxy[..., 0])
    y1 = torch.max(pred_xyxy[..., 1], gt_xyxy[..., 1])
    x2 = torch.min(pred_xyxy[..., 2], gt_xyxy[..., 2])
    y2 = torch.min(pred_xyxy[..., 3], gt_xyxy[..., 3])
    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    area_p = (pred_xyxy[..., 2] - pred_xyxy[..., 0]) * (pred_xyxy[..., 3] - pred_xyxy[..., 1])
    area_g = (gt_xyxy[..., 2] - gt_xyxy[..., 0]) * (gt_xyxy[..., 3] - gt_xyxy[..., 1])
    union = area_p + area_g - inter + 1e-6
    iou = inter / union

    cw = torch.max(pred_xyxy[..., 2], gt_xyxy[..., 2]) - torch.min(pred_xyxy[..., 0], gt_xyxy[..., 0])
    ch = torch.max(pred_xyxy[..., 3], gt_xyxy[..., 3]) - torch.min(pred_xyxy[..., 1], gt_xyxy[..., 1])
    c_area = cw * ch + 1e-6
    giou = iou - (c_area - union) / c_area
    giou_loss = (1 - giou).mean()

    # perda L1 para ajustar centro e tamanho
    l1_loss = torch.abs(pred_boxes - target_boxes).mean()
    return alpha * giou_loss + beta * l1_loss


# ===============================
# Fução para acompanhar IoU real dentro de run_epoch.
# ===============================
def mean_iou_batch(pred_boxes, target_boxes):
    # ambas em [cx,cy,w,h] normalizadas
    def cxcywh_to_xyxy(box):
        cx, cy, w, h = torch.chunk(box, 4, dim=-1)
        x1 = cx - 0.5 * w
        y1 = cy - 0.5 * h
        x2 = cx + 0.5 * w
        y2 = cy + 0.5 * h
        return torch.cat([x1, y1, x2, y2], dim=-1)

    pb = cxcywh_to_xyxy(pred_boxes)
    gb = cxcywh_to_xyxy(target_boxes)

    x1 = torch.max(pb[..., 0], gb[..., 0])
    y1 = torch.max(pb[..., 1], gb[..., 1])
    x2 = torch.min(pb[..., 2], gb[..., 2])
    y2 = torch.min(pb[..., 3], gb[..., 3])
    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    area_p = (pb[..., 2] - pb[..., 0]) * (pb[..., 3] - pb[..., 1])
    area_g = (gb[..., 2] - gb[..., 0]) * (gb[..., 3] - gb[..., 1])
    union = area_p + area_g - inter + 1e-6
    iou = (inter / union).clamp(0,1)
    return iou.mean().item()


# ===============================
# HIPERPARÂMETROS
# ===============================
BATCH_SIZE = config.BATCH_SIZE
LR = config.LR
EPOCHS = config.EPOCHS
PATCH_SIZE=config.PATCH_SIZE
D_MODEL=config.D_MODEL
NUM_HEADS=config.NUM_HEADS # D_MODEL=512 deve ser divisivel por NUM_HEADS=4
NUM_LAYERS=config.NUM_LAYERS
DIM_FEEDFORWARD=config.DIM_FEEDFORWARD 
DROPOUT=config.DROPOUT
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS=config.NUM_WORKERS
MAX_FRAMES_PER_SEQ = config.MAX_FRAMES_PER_SEQ
CHANNEL_TO_PLOT = config.CHANNEL_TO_PLOT

# ===============================
# TRANSFORMS
# ===============================
transform_vis = transforms.Compose([
    transforms.Resize((320, 256)), # 480, 270
    transforms.ToTensor()
])
transform_ir = transforms.Compose([
    transforms.Resize((320, 256)),
    transforms.ToTensor()
])
transform = {"visible": transform_vis, "infrared": transform_ir}

# ===============================
# DATASETS
# ===============================
train_dataset = AntiUAVRGBTDataset(
    root_dir="C:/Users/Micro/Documents/sourcecode/Anti-UAV-RGBT",
    split="train",
    transform=transform,
    filter_attributes=["OV", "VE", "OC"],
    max_frames_per_seq=MAX_FRAMES_PER_SEQ
)
val_dataset = AntiUAVRGBTDataset(
    root_dir="C:/Users/Micro/Documents/sourcecode/Anti-UAV-RGBT",
    split="val",
    transform=transform,
    filter_attributes=["OV", "VE", "OC"],
    max_frames_per_seq=MAX_FRAMES_PER_SEQ
)
test_dataset = AntiUAVRGBTDataset(
    root_dir="C:/Users/Micro/Documents/sourcecode/Anti-UAV-RGBT",
    split="test",
    transform=transform,
    filter_attributes=["OV", "VE", "OC"],
    max_frames_per_seq=MAX_FRAMES_PER_SEQ
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_multimodal)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_multimodal)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn_multimodal)

# ===============================
# MODELO
# ===============================
img_size_vis = (320, 256) #(480, 270)
img_size_ir  = (320, 256)
patch_size = PATCH_SIZE

model = VisionTransformerMultimodal(
    img_size_vis=img_size_vis,
    img_size_ir=img_size_ir,
    patch_size=patch_size, 
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT
).to(DEVICE)

# ===============================
# LOSS FUNCTIONS
# ===============================
box_loss_fn = combined_box_loss #nn.MSELoss()
exist_loss_fn = nn.BCELoss()

# ===============================
# OTIMIZADOR
# ===============================
optimizer = optim.AdamW(model.parameters(), lr=LR) # garantir que é executado depois do new model

# ===============================
# FUNÇÃO DE TREINO/VAL
# ===============================

def run_epoch(model, loader, optimizer=None, is_train=True):
    if is_train:
        model.train()
    else:
        model.eval()

    total_box_loss = 0
    total_exist_loss = 0
    total_attr_loss = 0
    total_samples = 0

    # antes do loop:
    iou_total = 0.0
    iou_count = 0

    with torch.set_grad_enabled(is_train):
        for batch_idx, batch in enumerate(tqdm(loader, desc="Treino" if is_train else "Val")):
            visible = batch["visible"].to(DEVICE)      # (B, T, C, H, W)
            infrared = batch["infrared"].to(DEVICE)   # (B, T, C, H, W)
            gt_boxes_vis = batch["gt_rect_vis"].to(DEVICE)  # GT visível
            gt_boxes_ir  = batch["gt_rect_ir"].to(DEVICE)   # GT infravermelho
            exists   = batch["exist"].float().to(DEVICE) # (B, T)
            
            outputs = model(visible, infrared, gt_boxes_vis, gt_boxes_ir, exists)
            pred_boxes_vis = outputs["pred_boxes_vis"]  # tensor no mesmo device
            pred_boxes_ir  = outputs["pred_boxes_ir"]   # tensor no mesmo device
            pred_exist = outputs["pred_exist"]

            # dentro do loop, logo depois de obter pred_boxes_vis e gt_boxes_vis:
            batch_iou = mean_iou_batch(pred_boxes_vis, gt_boxes_vis)
            iou_total += batch_iou * visible.size(0)  # pondera por batch size
            iou_count += visible.size(0)

            box_loss_vis = box_loss_fn(pred_boxes_vis, gt_boxes_vis)
            box_loss_ir  = box_loss_fn(pred_boxes_ir, gt_boxes_ir)
            box_loss = box_loss_vis + box_loss_ir
            exist_loss = exist_loss_fn(pred_exist, exists)

            #loss 
            loss = 1.0 * box_loss + 1.0 * exist_loss

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            batch_size = visible.size(0)
            total_box_loss += box_loss.item() * batch_size
            total_exist_loss += exist_loss.item() * batch_size
            total_samples += batch_size

            if batch_idx == 0:
                print("GT min/max:", gt_boxes_vis.min().item(), gt_boxes_vis.max().item())

            # 🧹 Libera tensores não mais usados
            del visible, infrared, gt_boxes_vis, gt_boxes_ir, exists, outputs
            torch.cuda.empty_cache()

    avg_box_loss = total_box_loss / total_samples
    avg_exist_loss = total_exist_loss / total_samples
    avg_attr_loss = total_attr_loss / total_samples if total_attr_loss > 0 else None

    # no final de run_epoch, calcule avg:
    avg_iou = iou_total / iou_count if iou_count>0 else 0.0
    # retorne também avg_iou
    
    return {
        "avg_box_loss": avg_box_loss,
        "avg_exist_loss": avg_exist_loss,
        "avg_attr_loss": avg_attr_loss,
        "avg_iou":avg_iou
    }

# ===============================
# TREINAMENTO COMPLETO COM CHECKPOINT
# ===============================
best_val_loss = float("inf")

for epoch in range(EPOCHS):
    print(f"\n=== Época {epoch+1}/{EPOCHS} ===")
    
    # Treino
    train_metrics = run_epoch(model, train_loader, optimizer, is_train=True)
    print(f"[TREINO] Box Loss: {train_metrics['avg_box_loss']:.4f} | Exist Loss: {train_metrics['avg_exist_loss']:.4f} | IoU: {train_metrics['avg_iou']:.3f}")

    # Validação
    with torch.no_grad():
        val_metrics = run_epoch(model, val_loader, is_train=False)
    print(f"[VAL] Box Loss: {val_metrics['avg_box_loss']:.4f} | Exist Loss: {val_metrics['avg_exist_loss']:.4f}")

    # Checkpoint do melhor modelo DEVE ser igual a mesma ponderação usada no treino (mudou a ponderação lá, mude aqui, para evitar inconsistencias).
    val_loss = 1.0 * val_metrics['avg_box_loss'] + 1.0 * val_metrics['avg_exist_loss']
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_loss": best_val_loss}, "best_model.pth")
        print("✅ Novo melhor modelo salvo!")
    
    # Limpeza de GPU, e memória por época.
    del train_metrics, val_metrics
    torch.cuda.empty_cache()
    gc.collect()




=== Época 1/40 ===


Treino:   0%|          | 0/3 [00:00<?, ?it/s]c:\Users\Micro\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Treino:  33%|███▎      | 1/3 [00:09<00:18,  9.16s/it]

GT min/max: 0.02031249925494194 0.8638888597488403


Treino: 100%|██████████| 3/3 [00:22<00:00,  7.61s/it]


[TREINO] Box Loss: 1.9279 | Exist Loss: 0.1596 | IoU: 0.009


...

[TREINO] Box Loss: 1.0643 | Exist Loss: 0.0043 | IoU: 0.134


Val:  50%|█████     | 1/2 [00:03<00:03,  3.86s/it]

GT min/max: 0.0 0.7962962985038757


Val: 100%|██████████| 2/2 [00:04<00:00,  2.16s/it]

[VAL] Box Loss: 1.0007 | Exist Loss: 0.0034


# TESTE

In [1]:
import torch
import cv2
import numpy as np
from tqdm import tqdm
import os
from dataloader import AntiUAVRGBTDataset, collate_fn_multimodal
from model import VisionTransformerMultimodal
from torchvision import transforms
from typing import Tuple
import config

# ===========================================================
# CONFIGURAÇÕES
# ===========================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_PATH = "output_videos"
os.makedirs(SAVE_PATH, exist_ok=True)

W_FINAL = 320 #480
H_FINAL = 256 #270

img_size_vis: Tuple[int, int] = (H_FINAL, W_FINAL)
img_size_ir: Tuple[int, int] = (320, 256)

PATCH_SIZE=config.PATCH_SIZE
D_MODEL=config.D_MODEL
NUM_HEADS=config.NUM_HEADS # D_MODEL=512 deve ser divisivel por NUM_HEADS=4
NUM_LAYERS=config.NUM_LAYERS
DIM_FEEDFORWARD=config.DIM_FEEDFORWARD 
DROPOUT=config.DROPOUT
MAX_FRAMES_PER_SEQ = 50 #config.MAX_FRAMES_PER_SEQ


# ===========================================================
# TRANSFORMS
# ===========================================================
transform_vis = transforms.Compose([
    transforms.Resize(img_size_vis),
    transforms.ToTensor()
])
transform_ir = transforms.Compose([
    transforms.Resize(img_size_ir),
    transforms.ToTensor()
])
transform = {"visible": transform_vis, "infrared": transform_ir}

# ===========================================================
# DATASET DE TESTE
# ===========================================================
test_dataset = AntiUAVRGBTDataset(
    root_dir="C:/Users/Micro/Documents/sourcecode/Anti-UAV-RGBT",
    split="test",
    transform=transform,
    filter_attributes=["OV", "VE", "OC"],
    max_frames_per_seq=MAX_FRAMES_PER_SEQ
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_fn_multimodal
)

# ===========================================================
# CARREGA MODELO
# ===========================================================
model = VisionTransformerMultimodal(
    img_size_vis=img_size_vis,
    img_size_ir=img_size_ir,
    patch_size=PATCH_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT
).to(DEVICE)

try:
    if not os.path.exists("best_model.pth"):
        torch.save({"model_state": model.state_dict(), "epoch": 0, "val_loss": 0.5}, "best_model.pth")
        print("AVISO: Checkpoint 'best_model.pth' não encontrado. Criado mock vazio.")
        
    checkpoint = torch.load("best_model.pth", map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    print("✅ Modelo carregado e pronto para teste.")
except Exception as e:
    print(f"ERRO ao carregar modelo: {e}. Usando pesos simulados/iniciais.")

model.eval()

# ===========================================================
# FUNÇÕES AUXILIARES
# ===========================================================
def draw_boxes(frame, gt_box, pred_box):
    h, w, _ = frame.shape
    if gt_box is not None:
        x, y, bw, bh = gt_box
        x1, y1 = int((x - bw/2) * w), int((y - bh/2) * h)
        x2, y2 = int((x + bw/2) * w), int((y + bh/2) * h)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, "GT", (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
    if pred_box is not None:
        x, y, bw, bh = pred_box
        x1, y1 = int((x - bw/2) * w), int((y - bh/2) * h)
        x2, y2 = int((x + bw/2) * w), int((y + bh/2) * h)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
        cv2.putText(frame, "Pred", (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)
    return frame

def compute_iou(boxA, boxB):
    if boxA is None or boxB is None:
        return 0.0
    ax1, ay1 = boxA[0]-boxA[2]/2, boxA[1]-boxA[3]/2
    ax2, ay2 = boxA[0]+boxA[2]/2, boxA[1]+boxA[3]/2
    bx1, by1 = boxB[0]-boxB[2]/2, boxB[1]-boxB[3]/2
    bx2, by2 = boxB[0]+boxB[2]/2, boxB[1]+boxB[3]/2
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(inter_x2 - inter_x1, 0)
    inter_h = max(inter_y2 - inter_y1, 0)
    inter_area = inter_w * inter_h
    union_area = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter_area
    return inter_area / union_area if union_area>0 else 0.0

def compute_center_error(boxA, boxB):
    if boxA is None or boxB is None:
        return 0.0
    return np.sqrt((boxA[0]-boxB[0])**2 + (boxA[1]-boxB[1])**2)

def compute_size_error(boxA, boxB):
    if boxA is None or boxB is None:
        return 0.0
    return np.sqrt((boxA[2]-boxB[2])**2 + (boxA[3]-boxB[3])**2)

# ===========================================================
# LOOP DE TESTE + GERAÇÃO DE VÍDEO COM MÉTRICAS
# ===========================================================
print("\nIniciando a Geração de Vídeos de Detecção...")
with torch.no_grad():
    for idx, batch in enumerate(tqdm(test_loader, desc="Gerando vídeos")):
        visible = batch["visible"].to(DEVICE)
        infrared = batch["infrared"].to(DEVICE)
        gt_boxes_vis = batch["gt_rect_vis"].to(DEVICE)
        exists = batch["exist"].float().to(DEVICE)

        outputs = model(visible, infrared, gt_boxes_vis, gt_boxes_vis, exists)
        pred_boxes_vis = outputs["pred_boxes_vis"].cpu().numpy()
        gt_boxes_vis_np = gt_boxes_vis.cpu().numpy()
        vis_frames = visible.cpu().numpy()[0]

        seq_name = f"sequence_{idx:03d}_visivel.mp4"
        out_path = os.path.join(SAVE_PATH, seq_name)
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out = cv2.VideoWriter(out_path, fourcc, 10, (W_FINAL, H_FINAL))

        # Inicializa métricas
        iou_list, center_err_list, size_err_list = [], [], []

        for t in range(vis_frames.shape[0]):
            frame = vis_frames[t].transpose(1,2,0)
            frame = (frame * 255).astype(np.uint8)
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

            gt_box = gt_boxes_vis_np[0, t] if t < gt_boxes_vis_np.shape[1] else None
            pred_box = pred_boxes_vis[0, t] if t < pred_boxes_vis.shape[1] else None

            # Calcula métricas
            iou_list.append(compute_iou(gt_box, pred_box))
            center_err_list.append(compute_center_error(gt_box, pred_box))
            size_err_list.append(compute_size_error(gt_box, pred_box))

            frame = draw_boxes(frame, gt_box, pred_box)
            out.write(frame)

        out.release()

        # Mostra métricas da sequência
        print(f"🎞️ {seq_name} | IoU médio: {np.mean(iou_list):.3f} | Center error médio: {np.mean(center_err_list):.3f} | Size error médio: {np.mean(size_err_list):.3f}")

print("\n✅ Todos os vídeos de teste foram gerados com sucesso!")


✅ Modelo carregado e pronto para teste.

Iniciando a Geração de Vídeos de Detecção...


Gerando vídeos:   0%|          | 0/18 [00:00<?, ?it/s]c:\Users\Micro\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Gerando vídeos:   6%|▌         | 1/18 [00:06<01:57,  6.89s/it]

🎞️ sequence_000_visivel.mp4 | IoU médio: 0.223 | Center error médio: 0.035 | Size error médio: 0.084


Gerando vídeos:  11%|█         | 2/18 [00:14<02:00,  7.50s/it]

🎞️ sequence_001_visivel.mp4 | IoU médio: 0.158 | Center error médio: 0.043 | Size error médio: 0.096


...

🎞️ sequence_015_visivel.mp4 | IoU médio: 0.234 | Center error médio: 0.055 | Size error médio: 0.103


Gerando vídeos:  94%|█████████▍| 17/18 [02:18<00:08,  8.08s/it]

🎞️ sequence_016_visivel.mp4 | IoU médio: 0.247 | Center error médio: 0.041 | Size error médio: 0.080


Gerando vídeos: 100%|██████████| 18/18 [02:26<00:00,  8.13s/it]

🎞️ sequence_017_visivel.mp4 | IoU médio: 0.392 | Center error médio: 0.043 | Size error médio: 0.031

✅ Todos os vídeos de teste foram gerados com sucesso!
